# Question 3a)

In [6]:
import matplotlib.pyplot as plt
import numpy as np
import numpy.linalg as lin
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve


def kappa_integral(x,y):
    p = (y - x)*(1 + 0.5*(x + y))
    return p

def build_massMatrix(N):
    # todo 3 b)
    h = 1/(N + 1)
    diagonal = np.ones(N)
    lower = np.ones(N - 1)
    upper = np.ones(N - 1)
    mass_matrix = (h/6) * sp.diags([lower, 4*diagonal, upper], offsets = [-1, 0, 1])
    return mass_matrix.toarray()


def build_rigidityMatrix(N):
    # todo 3 b)
    # Be careful with the indices!
    # kappa_integral could be helpful here
    
    M = np.zeros((N,N))
    h = 1/(N + 1)
    # The case of i=j
    for i in range(N):
        M[i,i] = (1 / (h**2)) * kappa_integral(i*h, (i+2)*h)

    # The case of i-j=1
    for i in range(1,N):
        M[i, i-1] = -(1 / (h**2)) * kappa_integral(i*h, (i+1)*h)


    # The case of j-i=1
    for i in range(N-1):
        M[i, i+1] = -(1 / (h**2)) * kappa_integral((i+1)*h, (i+2)*h)

    return M


def f(t,x):
    a = ((x + 1)*(np.pi**2) - 1) * np.exp(-t) * np.sin(np.pi * x)
    b = np.pi * np.exp(-t) * np.cos(np.pi*x)
    return a - b
    

def initial_value(x):
    return np.sin(np.pi*x)


def exact_solution_at_1(x):
    return np.exp(-1)*np.sin(np.pi*x)


def build_F(t,N):
    F = np.ones(N)
    h = 1/(N + 1)
    for i in range(N):
        a = f(t, ((i + 1)*h) - h/2)
        b = f(t, (i + 1)*h)
        c = f(t, ((i + 1)*h) + h/2)
        F[i] = (h/3)*(a + b + c)

    return F

def FEM_theta(N,M,theta):
    # get auxiliary variables
    h = 1/(N+1)
    k = 1/M

    # get u^0
    x_grid = np.linspace(h, N*h, N)
    um = initial_value(x_grid)

    #getting auxialiary matrices
    mass_matrix = build_massMatrix(N)
    A = build_rigidityMatrix(N)
    B = mass_matrix + k*theta*A

    # iterate
    j = 0
    for j in range(M):
        F_theta = theta*build_F((j+1)*k, N) + (1 - theta)*build_F(j*k, N)

        rhs = (mass_matrix - k*(1 - theta)* A) @ um + k*F_theta

        um += lin.solve(B, rhs)

    return um


